In [26]:
!pip install torch torch-geometric networkx scikit-learn

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
import random

from torch_geometric.utils import from_networkx
from torch_geometric.transforms import RandomLinkSplit
from sklearn.metrics import roc_auc_score

In [28]:
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [29]:
def create_grid_graph(n=20, feat_dim=32):
    G = nx.grid_2d_graph(n, n)
    G = nx.convert_node_labels_to_integers(G)

    for node in G.nodes():
        G.nodes[node]['x'] = np.random.randn(feat_dim)

    data = from_networkx(G)

    data.x = torch.tensor(
        np.array([G.nodes[i]['x'] for i in range(len(G.nodes()))]),
        dtype=torch.float
    )

    return data

data = create_grid_graph()
print(data)

Data(x=[400, 32], edge_index=[2, 1520])


In [30]:
transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=True
)

train_data, val_data, test_data = transform(data)

In [31]:
def compute_entropy(G, node):
    dist = nx.single_source_shortest_path_length(G, node)
    vals = list(dist.values())

    counts = np.bincount(vals)
    probs = counts / counts.sum()

    entropy = -sum(
        p * np.log(p + 1e-9)
        for p in probs if p > 0
    )

    return entropy

In [32]:
def build_anchor_sets(data):
    G = nx.Graph()

    edge_index = data.edge_index.cpu().numpy()

    for i in range(edge_index.shape[1]):
        G.add_edge(
            int(edge_index[0, i]),
            int(edge_index[1, i])
        )

    n = data.num_nodes
    num_sets = int(np.log2(n))

    entropy_scores = [
        compute_entropy(G, v)
        for v in range(n)
    ]

    ranked = np.argsort(entropy_scores)[::-1]

    anchor_sets = []

    for i in range(num_sets):
        c = max(1, n // (2 ** (i + 1)))
        anchor_sets.append(ranked[:c])

    return anchor_sets, G

In [33]:
anchor_sets, G = build_anchor_sets(train_data)

print([len(s) for s in anchor_sets])

[200, 100, 50, 25, 12, 6, 3, 1]


In [34]:
def compute_anchor_features(data, anchor_sets, G):
    n = data.num_nodes
    features = []

    for v in range(n):
        node_feat = []

        for anchors in anchor_sets:
            sims = []

            for a in anchors:
                try:
                    d = nx.shortest_path_length(G, v, int(a))
                    sims.append(1.0 / (d + 1))
                except:
                    sims.append(0.0)

            node_feat.append(np.mean(sims))

        features.append(node_feat)

    return torch.tensor(features, dtype=torch.float)

In [35]:
anchor_feat = compute_anchor_features(
    train_data,
    anchor_sets,
    G
)

print(anchor_feat.shape)

torch.Size([400, 8])


In [36]:
class IEA_GNN_F_2L(nn.Module):
    def __init__(self, in_dim, anchor_dim, hidden=64):
        super().__init__()

        self.lin1 = nn.Linear(
            in_dim + anchor_dim,
            hidden
        )

        self.lin2 = nn.Linear(
            hidden,
            hidden
        )

        self.dropout = nn.Dropout(0.5)

    def aggregate(self, x, edge_index):
        row, col = edge_index

        out = torch.zeros_like(x)
        out.index_add_(0, row, x[col])

        deg = torch.bincount(
            row,
            minlength=x.size(0)
        ).float().unsqueeze(1)

        deg[deg == 0] = 1

        return out / deg

    def forward(self, x, edge_index):
        h = self.aggregate(x, edge_index)
        h = F.relu(self.lin1(h))
        h = self.dropout(h)

        h = self.aggregate(h, edge_index)
        h = self.lin2(h)

        return h

In [37]:
model = IEA_GNN_F_2L(
    train_data.x.shape[1],
    anchor_feat.shape[1],
    hidden=64
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

train_data = train_data.to(device)
test_data = test_data.to(device)
anchor_feat = anchor_feat.to(device)

In [38]:
def decode(z, edge_index):
    return (
        z[edge_index[0]] *
        z[edge_index[1]]
    ).sum(dim=1)

In [39]:
for epoch in range(2500):
    model.train()
    optimizer.zero_grad()

    x = torch.cat(
        [train_data.x, anchor_feat],
        dim=1
    )

    z = model(x, train_data.edge_index)

    pred = decode(
        z,
        train_data.edge_label_index
    )

    loss = F.binary_cross_entropy_with_logits(
        pred,
        train_data.edge_label.float()
    )

    loss.backward()
    optimizer.step()

    if epoch % 250 == 0:
        print(epoch, loss.item())

0 0.7547944784164429
250 0.1373610943555832
500 0.13484418392181396
750 0.09151376038789749
1000 0.045039527118206024
1250 0.06579554826021194
1500 0.055523961782455444
1750 0.08672124147415161
2000 0.045015882700681686
2250 0.08511018753051758


In [40]:
test_anchor_sets, test_G = build_anchor_sets(test_data)

test_anchor_feat = compute_anchor_features(
    test_data,
    test_anchor_sets,
    test_G
).to(device)

model.eval()

x_test = torch.cat(
    [test_data.x, test_anchor_feat],
    dim=1
)

with torch.no_grad():
    z = model(x_test, test_data.edge_index)

pred = torch.sigmoid(
    decode(z, test_data.edge_label_index)
).cpu().numpy()

labels = test_data.edge_label.cpu().numpy()

auc = roc_auc_score(labels, pred)

print("IEA-GNN-F-2L AUC:", auc)

IEA-GNN-F-2L AUC: 0.6597991689750693
